In [ ]:
import pandas as pd
import numpy as np
import pyomo.environ as pyo
import matplotlib.pyplot as plt

!pip install pyomo
# and some LP/MILP solver available on your system

!apt-get update
!apt-get install -y coinor-cbc



Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,020 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [77.8 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,070 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]
Get:14 ht

# Add forecasts to dataset

In [ ]:
df_merged = pd.read_excel('final_dataset_MILP.xlsx')
# Make sure timestamp is datetime
df_merged["utc_timestamp"] = pd.to_datetime(df_merged["utc_timestamp"])

# Columns with actual load and PV
load_col = "Appliances_load"
pv_col = "pv"

# Define which year is used for each monthly profile
profile_year_by_month = {
    1: 2024,
    2: 2024,
    3: 2024,
    4: 2024,
    5: 2024,
    6: 2024,
    7: 2024,
    8: 2024,
    9: 2024,
    10: 2022,
    11: 2022,
    12: 2022,
}

# Helper columns
df_merged["month"] = df_merged["utc_timestamp"].dt.month
df_merged["year"] = df_merged["utc_timestamp"].dt.year
df_merged["hour"] = df_merged["utc_timestamp"].dt.hour

# Create empty columns
df_merged["Load_profile"] = pd.NA
df_merged["PV_profile"] = pd.NA

# Build and assign each monthly profile
for month, profile_year in profile_year_by_month.items():

    profile_data = df_merged[
        (df_merged["year"] == profile_year) &
        (df_merged["month"] == month)
    ]

    hourly_profile = (
        profile_data
        .groupby("hour")[[load_col, pv_col]]
        .mean()
        .rename(columns={
            load_col: "Load_profile",
            pv_col: "PV_profile"
        })
    )

    rows_this_month = df_merged["month"] == month

    df_merged.loc[rows_this_month, "Load_profile"] = (
        df_merged.loc[rows_this_month, "hour"]
        .map(hourly_profile["Load_profile"])
    )

    df_merged.loc[rows_this_month, "PV_profile"] = (
        df_merged.loc[rows_this_month, "hour"]
        .map(hourly_profile["PV_profile"])
    )

# Optional: remove helper columns
df_merged = df_merged.drop(columns=["month", "year", "hour"])



# Define period to evaluate


In [ ]:
is_period = True #True when its a continuous period, False when its particular dates not (not continuous period)

if is_period:

  DAY = 30+31+30+31-2 #Jan 1st 2023
  START_HOUR = 0
  PERIOD = 1 # days
  start_idx = DAY * 24 + START_HOUR   # day * hour
  df_day = df_merged.iloc[start_idx : start_idx + (24 * PERIOD)].copy()

else:

  # Ensure datetime format
  df_merged['utc_timestamp'] = pd.to_datetime(df_merged['utc_timestamp'])

  #Dates in DD MM YYYY format
  selected_dates = [
      '02 01 2023', '02 02 2023', '15 03 2023', '03 04 2023', '31 05 2023', '01 06 2023', '24 08 2023',
                      '06 09 2023', '10 10 2023', '29 11 2023', '22 01 2023', '14 02 2023', '06 03 2023',
                      '30 08 2023', '14 09 2023', '23 10 2023', '06 12 2023'
  ]


  # Filter dataframe
  df_day = df_merged[
      df_merged['utc_timestamp'].dt.strftime('%d %m %Y').isin(selected_dates)
  ].copy()

  # Reset index
  df_day = df_day.reset_index(drop=True)



# Run MILP with the forecasts and determine the optimal actions

In [ ]:


#### -----------------Case for no PV nor Battery --------------#####

df_no_pv_no_batt = df_day.copy()
df_no_pv_no_batt['EV_assumption'] = 0.0
df_no_pv_no_batt.loc[df_no_pv_no_batt['utc_timestamp'].dt.hour.between(5, 8), 'EV_assumption'] = 3.6

#### ------------------------------ MILP ------------------------###
# Ensure sorted and continuous hourly index
df_day = df_day.sort_values('utc_timestamp').reset_index(drop=True)
# Build simple dicts keyed by time index (0..N-1)
T_index = df_day.index.tolist()
demand = df_day['Load_profile'].to_dict()
pv_gen = df_day['PV_profile'].to_dict()
price_buy = df_day['price_EUR_kWh'].to_dict()

sell_factor = 0.8
price_sell = {t: sell_factor * price_buy[t] for t in T_index}
initial_price = demand[0] * price_buy[0]
import pyomo.environ as pyo




# -----------------------
# 1. Create model & set
# -----------------------
model = pyo.ConcreteModel()
model.T = pyo.Set(initialize=T_index, ordered=True)  # time steps (0..N-1)










# -----------------------
# 2. Parameters
# -----------------------
# Data from pandas
model.demand = pyo.Param(model.T, initialize=demand)      # kWh
model.pv      = pyo.Param(model.T, initialize=pv_gen)     # kWh
model.price_buy  = pyo.Param(model.T, initialize=price_buy)   # EUR/kWh
model.price_sell = pyo.Param(model.T, initialize=price_sell)  # EUR/kWh
# Tech parameters (you choose these)
Cap_batt   = 15.0   # kWh
P_batt_max = 5.0    # kW (kWh per hour)
Batt_init = Cap_batt/2   # initial SoC

#-----------------  CHECK PARAMETERS BECAUSE OF SOC ASSUMPTIONS ----------------------------------###
model.Cap_batt   = pyo.Param(initialize=Cap_batt)
model.P_batt_max = pyo.Param(initialize=P_batt_max)
model.Batt_init     = pyo.Param(initialize=Batt_init)
model.Init_import = pyo.Param(initialize=demand[0])

P_grid_max = 20 #kW
model.P_grid_max = pyo.Param(initialize=P_grid_max)

# -----------------------
# 3. Decision variables
# -----------------------
model.P_grid_import = pyo.Var(model.T, domain=pyo.NonNegativeReals)
model.P_grid_export = pyo.Var(model.T, domain=pyo.NonNegativeReals)
model.P_batt_ch  = pyo.Var(model.T, domain=pyo.NonNegativeReals)
model.P_batt_dis = pyo.Var(model.T, domain=pyo.NonNegativeReals)
model.SOC_batt = pyo.Var(model.T, domain=pyo.NonNegativeReals, bounds=(0, Cap_batt))

# (Optional) binaries to avoid simultaneous buy & sell
model.y_buy = pyo.Var(model.T, domain=pyo.Binary)








# -----------------------
# 4. Constraints
# -----------------------
# 4.1 Energy balance
def energy_balance_rule(m, t):
    return ( m.P_grid_import[t]
          + m.P_batt_dis[t]
          #+ m.P_EV_dis[t]  #V2G
          + m.pv[t]
          ==
            m.demand[t]
          + m.P_batt_ch[t]
          #+ m.P_EV_ch[t]
          + m.P_grid_export[t]
    )
model.EnergyBalance = pyo.Constraint(model.T, rule=energy_balance_rule)

# 4.2 Battery power limits
def batt_ch_limit_rule(m, t):
    return m.P_batt_ch[t] <= m.P_batt_max

def batt_dis_limit_rule(m, t):
    return m.P_batt_dis[t] <= m.P_batt_max

model.BattChLimit  = pyo.Constraint(model.T, rule=batt_ch_limit_rule)
model.BattDisLimit = pyo.Constraint(model.T, rule=batt_dis_limit_rule)
# 4.3 Battery SoC dynamics
def batt_soc_rule(m, t):
    if t == m.T.first():
        # initial condition
        return m.SOC_batt[t] == m.Batt_init #\
    else:
        t_prev = m.T.prev(t)
        return m.SOC_batt[t] == m.SOC_batt[t_prev] \
              + m.P_batt_ch[t] \
              - m.P_batt_dis[t]

model.BattSOC = pyo.Constraint(model.T, rule=batt_soc_rule)

# 4.5 Grid power limit & buy/sell exclusivity (MILP part)
def grid_import_limit_rule(m, t):
    if t == 0:
        return m.P_grid_import[t] == m.Init_import
    else:
        return m.P_grid_import[t] <= m.y_buy[t] * m.P_grid_max

def grid_export_limit_rule(m, t):
    if t == 0:
        return m.P_grid_export[t] == 0
    else:
        return m.P_grid_export[t] <= (1 - m.y_buy[t]) * m.P_grid_max

model.GridImportLimit = pyo.Constraint(model.T, rule=grid_import_limit_rule)
model.GridExportLimit = pyo.Constraint(model.T, rule=grid_export_limit_rule)






# -----------------------
# 5. Objective
# -----------------------
def objective_rule(m):
  return sum(
      m.price_buy[t] * m.P_grid_import[t]
      - m.price_sell[t] * m.P_grid_export[t]
      for t in m.T
  )
model.Obj = pyo.Objective(rule=objective_rule, sense=pyo.minimize)






# -----------------------
# 6. Solve
# -----------------------
solver = pyo.SolverFactory("highs")
result = solver.solve(model, tee=True)

milp_spent = np.round(pyo.value(model.Obj), 4)
print("Total money spent in that period:", milp_spent, ' sek')


results = pd.DataFrame({
    "utc_timestamp": df_day["utc_timestamp"],
    "P_batt_ch":     [pyo.value(model.P_batt_ch[t]) for t in model.T],
    "P_batt_dis":    [pyo.value(model.P_batt_dis[t]) for t in model.T],
    "SOC_batt":      [pyo.value(model.SOC_batt[t]) for t in model.T],
    'PV_Available':  [pyo.value(model.pv[t]) for t in model.T],
    'Load':          [pyo.value(model.demand[t]) for t in model.T],
    'Price_buy':        [pyo.value(model.price_buy[t]) for t in model.T],
    'Price_sell':        [pyo.value(model.price_sell[t]) for t in model.T]})



Running HiGHS 1.14.0 (git hash: 7df0786): Copyright (c) 2026 under MIT licence terms
MIP has 144 rows; 143 cols; 331 nonzeros; 23 integer variables (23 binary)
Coefficient ranges:
  Matrix  [1e+00, 2e+01]
  Cost    [8e-01, 2e+00]
  Bound   [1e+00, 2e+01]
  RHS     [1e+00, 2e+01]
Presolving model
92 rows, 138 cols, 275 nonzeros 0s
91 rows, 114 cols, 227 nonzeros 0s
Presolve reductions: rows 91(-53); columns 114(-29); nonzeros 227(-104) 

Solving MIP model with:
   91 rows
   114 cols (23 binary, 0 integer, 0 implied int., 91 continuous, 0 domain fixed)
   227 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |         

#Compute the actual cost when following the actions determined

In [ ]:
results['Load_real'] = df_day['Appliances_load']
results['pv_real'] = df_day['pv']
# Compute net battery power
# (charging consumes power, discharging supplies power)
results["P_batt"] = results["P_batt_ch"] - results["P_batt_dis"]

# Compute net energy demand
results["net_energy_demand"] = (
    results["Load_real"]
    - results["pv_real"]
    + results["P_batt"]
)

# Compute grid import / export
results["P_grid_import"] = results["net_energy_demand"].clip(lower=0)

results["P_grid_export"] = (-results["net_energy_demand"]).clip(lower=0)

results['Cost'] =  results["P_grid_import"] * results["Price_buy"] - results["P_grid_export"] * results["Price_sell"]
print('Total cost: ', np.round(results['Cost'].sum(), 4), ' sek')

Total cost:  138.5039  sek
